# Biomarker S1 — Ghép cohort: KL grade + ảnh DESS + trạng thái mask

Mục tiêu notebook này: dựng **bảng cohort** (`case_id, subject, visit, side, KL, dess_path, mask_status`)
để 2 notebook sau (`S2` inference, `S3` extract biomarker) chạy thẳng, không phải tra cứu lại.

**Đọc kỹ trước khi chạy:**
- `case_id` dùng đúng quy ước đã có trong pipeline segmentation: `{subject}_{visit}_{side}`
  (vd `9003406_V00_L`), khớp với cách `convert_imorphics_to_nnunet.ipynb` đặt tên.
- 176 ca **iMorphics** (`Dataset012_iMorphics`, 140 train + 36 test) giữ **subject ID thật** của OAI
  → join thẳng được với bảng KL grade.
- **[CẬP NHẬT]** 507 ca **OAIZIB-CM** (`Dataset001_KneeOA`, 404 train + 103 test) **CÓ mapping sang
  subject ID thật** qua `subInfo_train.xlsx` (+ `subInfo_test.xlsx` nếu có) — cột `SubjectID` khớp
  `CMT-ID` dùng trong tên file `oaizib_XXX`. Giả định cũ ("507 ca ZIB không map được, chỉ dùng cho
  segmentation") đã bị loại bỏ — notebook này giờ **tự động gộp thêm 507 ca này vào cohort
  classification** ở bước 3b bên dưới.
- Nguồn công bố OAI-ZIB (Ambellan et al. 2019) xác nhận: toàn bộ 507 ca đều là **đầu gối PHẢI**,
  chụp ở **visit baseline (V00)** — nên `case_id` cho các ca này luôn suy ra được dạng
  `{SubjectID}_V00_R`, không cần file manifest riêng, không cần đoán visit.
- Ảnh DESS gốc của **pool OAI_DESS rộng hơn** (tải theo barcode, ví dụ từ `label.csv`) **CHƯA dùng
  được** ở thời điểm viết notebook này: (1) `OAI_DESS` trên Drive hiện chỉ có đủ 176 file (khớp
  iMorphics), ảnh full-res cho phần còn lại chưa tải; (2) `label.csv` không có cột `visit` nên chưa
  thể suy ra `case_id` an toàn. Phần `MANIFEST_FILE` ở bước 3 vẫn giữ lại (không xoá) để dùng sau
  này khi 2 vấn đề trên được giải quyết — hiện tại nó sẽ không có tác dụng nếu bạn để `None`.


In [1]:
!pip install -q pandas nibabel openpyxl

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 0) Cấu hình — CHỈNH các đường dẫn cho khớp máy bạn

In [3]:
from pathlib import Path
import pandas as pd

# >>> nnUNet_raw root — TU DONG DO TIM (fix loi "khong ton tai" khi path hardcode sai) <<<
#     Neu ban DA BIET chinh xac duong dan, gan thang RAW_OVERRIDE = Path("...") de bo qua do tim.
RAW_OVERRIDE = None   # vd: Path("/content/drive/MyDrive/nnUNet_raw")

def _find_raw_dir():
    if RAW_OVERRIDE is not None:
        return Path(RAW_OVERRIDE)
    search_roots = []
    mydrive = Path("/content/drive/MyDrive")
    if mydrive.exists():
        search_roots.append(mydrive)
    shared_root = Path("/content/drive/Shareddrives")
    if shared_root.exists():
        search_roots += [d for d in shared_root.iterdir() if d.is_dir()]
    for root in search_roots:
        hits = sorted(root.rglob("Dataset012_iMorphics"))
        if hits:
            return hits[0].parent
    return None

_raw_found = _find_raw_dir()
if _raw_found is None:
    print("CANH BAO: khong tu dong tim thay thu muc chua \'Dataset012_iMorphics\' trong")
    print("  /content/drive/MyDrive hoac /content/drive/Shareddrives.")
    print("  Kha nang: (1) dang mount SAI tai khoan Google so voi tai khoan da chay cac")
    print("  notebook convert_*/merge_s3_s4_assemble; (2) du lieu nam trong Shared Drive")
    print("  can \'Add shortcut to Drive\' truoc; (3) merge_s3_s4_assemble chua chay xong")
    print("  (Dataset020_KneeUnion chua duoc tao). Dat RAW_OVERRIDE = Path(\'duong dan that\')")
    print("  o tren neu ban biet chinh xac path, roi chay lai cell nay.")
    RAW = Path("/content/drive/MyDrive/nnUNet_raw")   # fallback, cac buoc sau se bao ro neu vAn thieu
else:
    RAW = _raw_found
    print("Tu dong tim thay RAW =", RAW)

D001 = RAW / "Dataset001_KneeOA"      # OAI-ZIB (bone+cart), ID noi bo oaizib_XXX
D012 = RAW / "Dataset012_iMorphics"   # iMorphics, subject ID that, 140 train + 36 test
D020 = RAW / "Dataset020_KneeUnion"   # 8-class GT+pseudo da co san (544 ca)

for name, d in [("D001", D001), ("D012", D012), ("D020", D020)]:
    print(f"{name} = {d}  | ton tai: {d.exists()}")

DESS_DIR = Path("/content/drive/MyDrive/OAI_DESS")   # anh DESS goc (pool rong hon), dat ten theo barcode

# >>> KL grade file (OAI semi-quantitative X-ray reading) <<<
#     Ban da xac nhan: file nay nam trong OAICompleteData_ASCII (da giai nen) tren Drive,
#     nhung duong dan chinh xac (thu muc con) chua ro -> TU DONG DO TIM ben duoi.
#     Neu ban da biet chinh xac path, co the bo qua auto-search: gan thang KL_FILE = Path("...")
#     va set KL_FILE_OVERRIDE = KL_FILE (dong ben duoi).
KL_FILE_OVERRIDE = None   # DA XAC NHAN: file that nam truc tiep tai
#   /content/drive/MyDrive/OAICompleteData_ASCII/KXR_SQ_BU00.txt (khong co thu muc con
#   "OAI Complete Data_ASCII" long ben trong nhu ban dau tuong -- thu muc con do THUC RA
#   chua anh DESS (.nii.gz/.json theo barcode), khong phai bang KL). Dat None de dung
#   auto-search (rglob) ben duoi, tu tim dung file that.
KL_ID_COL, KL_SIDE_COL, KL_GRADE_COL = "ID", "SIDE", "V00XRKL"
KL_VISIT = "V00"   # doi neu ban dung file cua visit khac (vd kxr_sq_bu01.txt -> "V01"...)

if KL_FILE_OVERRIDE is not None:
    KL_FILE = Path(KL_FILE_OVERRIDE)
else:
    _drive_root = Path("/content/drive/MyDrive")
    _hits = sorted(_drive_root.rglob("OAICompleteData_ASCII/OAI Complete Data_ASCII/KXR_SQ_BU00.txt"))
    if len(_hits) == 0:
        # thu tim khong phan biet hoa/thuong, phong khi ten file hoi khac
        _hits = sorted(p for p in _drive_root.rglob("*") if p.is_file() and "kxr_sq_bu00" in p.name.lower())
    if len(_hits) == 0:
        raise FileNotFoundError(
            "Khong tu dong tim thay kxr_sq_bu00.txt trong /content/drive/MyDrive. "
            "Kiem tra lai thu muc OAICompleteData_ASCII da giai nen dung chua, "
            "hoac gan thang KL_FILE_OVERRIDE = Path(\'duong dan that\') o tren."
        )
    elif len(_hits) > 1:
        print("CANH BAO: tim thay nhieu file kxr_sq_bu00.txt, dang dung file dau tien:")
        for h in _hits:
            print("  -", h)
    KL_FILE = _hits[0]

print("KL_FILE dang dung:", KL_FILE)

# >>> CHINH (neu co): manifest barcode -> (subject, visit, side) cho toan bo pool OAI_DESS <<<
#     Neu ban chua co file nay, dat MANIFEST_FILE = None -> notebook chi dung 176 ca iMorphics
#     + 507 ca OAIZIB-CM (buoc 3b). Khi ban tai them anh full-res + xac dinh duoc visit cho
#     cac barcode trong label.csv, quay lai dien MANIFEST_FILE de mo rong them.
MANIFEST_FILE = None   # vd: Path("/content/drive/MyDrive/OAI_clinical/image_manifest.csv")
MANIFEST_COLS = dict(barcode="barcode", subject="subject", visit="visit", side="side")

# >>> MOI: subInfo cua OAIZIB-CM (co SubjectID that + CMT-ID noi bo) <<<
SUBINFO_TRAIN = Path("/content/drive/MyDrive/subInfo_train.xlsx")   # sua path cho khop may ban
SUBINFO_TEST  = Path("/content/drive/MyDrive/subInfo_test.xlsx")    # None hoac duong dan neu co

OUT_DIR = Path("/content/drive/MyDrive/knee_biomarkers")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUT_DIR =", OUT_DIR)

Tu dong tim thay RAW = /content/drive/MyDrive/nnUNet_raw
D001 = /content/drive/MyDrive/nnUNet_raw/Dataset001_KneeOA  | ton tai: True
D012 = /content/drive/MyDrive/nnUNet_raw/Dataset012_iMorphics  | ton tai: True
D020 = /content/drive/MyDrive/nnUNet_raw/Dataset020_KneeUnion  | ton tai: True
KL_FILE dang dung: /content/drive/MyDrive/OAICompleteData_ASCII/OAI Complete Data_ASCII/KXR_SQ_BU00.txt
OUT_DIR = /content/drive/MyDrive/knee_biomarkers


## 1) Load bảng KL grade

OAI công bố dạng ASCII tab/`|`-delimited. Code dưới đọc linh hoạt (tab hoặc dấu phẩy), và
chuẩn hoá `SIDE` (1/2) → (R/L). **Nếu tên cột ở file bạn khác** `KL_ID_COL/KL_SIDE_COL/KL_GRADE_COL`
ở trên, sửa lại rồi chạy lại cell này — đừng đoán, in `df_kl_raw.columns` ra để kiểm tra trước.

In [4]:
def read_oai_table(path):
    for sep in ["\t", "|", ","]:
        try:
            df = pd.read_csv(path, sep=sep, engine="python")
            if df.shape[1] > 1:
                return df
        except Exception:
            continue
    raise ValueError(f"Khong doc duoc {path} voi cac dau phan cach thu (tab/|/,)")

def parse_oai_code(series):
    """
    Cac cot OAI ASCII thuong ma hoa dang "1: Right", "2: Left", "0: 0", "2: 2"...
    (label gan sau dau \':\'). pd.to_numeric truc tiep se ra NaN toan bo vi khong
    phai so thuan. Ham nay tach lay MA SO (phan truoc dau \':\') roi moi ep so.
    Neu gia tri da la so thuan (khong co \':\') thi van hoat dong binh thuong.
    """
    s = series.astype(str).str.strip()
    code = s.str.extract(r"^\s*(-?\d+)", expand=False)
    return pd.to_numeric(code, errors="coerce")

assert KL_FILE.exists(), f"KHONG THAY file KL grade: {KL_FILE} — chinh KL_FILE o cell config"
df_kl_raw = read_oai_table(KL_FILE)
print("cot trong file KL:", list(df_kl_raw.columns)[:20])

SIDE_MAP = {1: "R", 2: "L"}
_side_code = parse_oai_code(df_kl_raw[KL_SIDE_COL])       # "1: Right" -> 1, "2: Left" -> 2
_kl_code = parse_oai_code(df_kl_raw[KL_GRADE_COL])         # "2: 2" -> 2, "0: 0" -> 0

df_kl = pd.DataFrame({
    "subject": df_kl_raw[KL_ID_COL].astype(str),
    "side":    _side_code.map(SIDE_MAP),
    "visit":   KL_VISIT,
    "KL":      _kl_code,
}).dropna(subset=["KL", "side"])
df_kl["KL"] = df_kl["KL"].astype(int)
df_kl["case_id"] = df_kl["subject"] + "_" + df_kl["visit"] + "_" + df_kl["side"]
print("so ban ghi KL grade hop le:", len(df_kl))
print(df_kl["KL"].value_counts().sort_index())
df_kl.head()

cot trong file KL: ['ID', 'SIDE', 'READPRJ', 'VERSION', 'V00BARCDBU', 'V00XROSFM', 'V00XRSCFM', 'V00XRCYFM', 'V00XRJSM', 'V00XRCHM', 'V00XROSTM', 'V00XRSCTM', 'V00XRCYTM', 'V00XRATTM', 'V00XRKL', 'V00XROSFL', 'V00XRSCFL', 'V00XRCYFL', 'V00XRJSL', 'V00XRCHL']
so ban ghi KL grade hop le: 16592
KL
0    8085
1    3575
2    3152
3    1466
4     314
Name: count, dtype: int64


,subject,side,visit,KL,case_id
0,9000099,R,V00,2,9000099_V00_R
1,9000099,L,V00,3,9000099_V00_L
2,9000296,R,V00,2,9000296_V00_R
3,9000296,L,V00,3,9000296_V00_L
4,9000622,R,V00,1,9000622_V00_R


In [5]:
# Kiem tra xem 1 case_id co bi nhieu READPRJ doc KL khac nhau khong truoc khi dedupe
dup_check = df_kl.groupby("case_id")["KL"].nunique()
n_conflict = (dup_check > 1).sum()
print(f"so case_id co KL grade MAU THUAN giua cac READPRJ khac nhau: {n_conflict}")
if n_conflict > 0:
    print("vi du:", dup_check[dup_check > 1].index[:5].tolist())
    # can quyet dinh nguon nao dang tin hon (vd uu tien READPRJ == 15 neu la du an doc chinh)
    # in ra de kiem tra truoc khi quyet dinh drop nguon nao

# Neu KHONG co mau thuan (cung case_id, KL giong het nhau o moi dong) -> dedupe an toan:
before = len(df_kl)
df_kl = df_kl.drop_duplicates(subset="case_id", keep="first").reset_index(drop=True)
print(f"df_kl: {before} -> {len(df_kl)} (sau dedupe theo case_id)")

so case_id co KL grade MAU THUAN giua cac READPRJ khac nhau: 54
vi du: ['9001897_V00_L', '9002316_V00_L', '9043446_V00_R', '9043945_V00_R', '9066677_V00_L']
df_kl: 16592 -> 8953 (sau dedupe theo case_id)


In [6]:
from pathlib import Path

print("=== 1) KL file — 3 dong dau tho (raw) ===")
with open(KL_FILE, "r", errors="replace") as f:
    for i, line in zip(range(3), f):
        print(repr(line[:300]))

print("\n=== 2) Dataset020/labelsTr ===")
print("ton tai:", D020.exists(), "|", (D020/'labelsTr').exists())
if (D020/'labelsTr').exists():
    print("so file trong labelsTr:", len(list((D020/'labelsTr').glob('*'))))
    print("vi du:", list((D020/'labelsTr').glob('*'))[:5])

print("\n=== 3) Dataset012 imagesTr/imagesTs ===")
for sub in ["imagesTr", "imagesTs"]:
    p = D012/sub
    print(sub, "ton tai:", p.exists())
    if p.exists():
        print("  so file:", len(list(p.glob('*'))), "| vi du:", list(p.glob('*'))[:3])

print("\n=== 4) Dataset001 imagesTr/imagesTs (OAIZIB-CM) ===")
for sub in ["imagesTr", "imagesTs"]:
    p = D001/sub
    print(sub, "ton tai:", p.exists())
    if p.exists():
        print("  so file:", len(list(p.glob('*'))), "| vi du:", list(p.glob('*'))[:3])

print("\n=== 5) nnUNet_raw co gi ===")
print(list(RAW.glob('*')) if RAW.exists() else "RAW khong ton tai")

=== 1) KL file — 3 dong dau tho (raw) ===
'ID|SIDE|READPRJ|VERSION|V00BARCDBU|V00XROSFM|V00XRSCFM|V00XRCYFM|V00XRJSM|V00XRCHM|V00XROSTM|V00XRSCTM|V00XRCYTM|V00XRATTM|V00XRKL|V00XROSFL|V00XRSCFL|V00XRCYFL|V00XRJSL|V00XRCHL|V00XROSTL|V00XRSCTL|V00XRCYTL|V00XRATTL\n'
'9000099|1: Right|15|0.9|016600839603|0: 0|0: 0|0: 0|0|0: 0|1: 1|0: 0|0: 0|0: 0|2: 2|2: 2|0: 0|0: 0|0|0: 0|1: 1|0: 0|0: 0|0: 0\n'
'9000099|2: Left|15|0.9|016600839603|0: 0|0: 0|0: 0|0|0: 0|0: 0|0: 0|1: 1|0: 0|3: 3|2: 2|2: 2|0: 0|2|0: 0|1: 1|2: 2|0: 0|0: 0\n'

=== 2) Dataset020/labelsTr ===
ton tai: True | True
so file trong labelsTr: 541
vi du: [PosixPath('/content/drive/MyDrive/nnUNet_raw/Dataset020_KneeUnion/labelsTr/oaizib_004.nii.gz'), PosixPath('/content/drive/MyDrive/nnUNet_raw/Dataset020_KneeUnion/labelsTr/oaizib_002.nii.gz'), PosixPath('/content/drive/MyDrive/nnUNet_raw/Dataset020_KneeUnion/labelsTr/oaizib_001.nii.gz'), PosixPath('/content/drive/MyDrive/nnUNet_raw/Dataset020_KneeUnion/labelsTr/oaizib_003.nii.gz'), Pos

## 2) Danh sách case đã có mask sẵn (Dataset020, GT+pseudo — dùng lại, KHÔNG cần infer)

In [7]:
labels_dir = D020 / "labelsTr"
done_case_ids = sorted(p.stem.replace(".nii","") for p in labels_dir.glob("*.nii.gz"))
# Chi giu nhung id dung dinh dang {subject}_{visit}_{side} (nguon goc iMorphics that);
# id kieu "oaizib_XXX" se khong khop bang KL (khong co subject that) nen bi loai o buoc merge.
print("tong ca trong Dataset020:", len(done_case_ids))
print("vi du 5 id:", done_case_ids[:5])

tong ca trong Dataset020: 541
vi du 5 id: ['9007827_V00_L', '9007827_V01_L', '9040390_V00_R', '9040390_V01_R', '9047800_V00_L']


## 3) (Tùy chọn) Manifest mở rộng cohort ngoài 176 ca iMorphics

Nếu bạn có `MANIFEST_FILE` (barcode → subject/visit/side) cho toàn bộ pool `OAI_DESS`,
cell này build bảng `df_dess` đầy đủ. Nếu không có, chỉ dùng ảnh đã có sẵn trong
`Dataset012_iMorphics` (176 ca) — vẫn chạy được, chỉ là cohort nhỏ hơn.

**[CẬP NHẬT]** Hiện tại `OAI_DESS` trên Drive chỉ có đủ 176 file (khớp iMorphics), nên dù
`label.csv` có cấu trúc đúng như 1 manifest (barcode trích từ cột `mri_path`), nó **chưa dùng
được** ở đây — chưa có ảnh full-res tương ứng, và chưa rõ visit. Giữ nguyên logic
`MANIFEST_FILE` để dùng sau; cohort thật sự được mở rộng ở bước **3b** ngay dưới.

In [8]:
if MANIFEST_FILE is not None and Path(MANIFEST_FILE).exists():
    dfm = read_oai_table(MANIFEST_FILE)
    df_dess = pd.DataFrame({
        "subject": dfm[MANIFEST_COLS["subject"]].astype(str),
        "visit":   dfm[MANIFEST_COLS["visit"]].astype(str),
        "side":    dfm[MANIFEST_COLS["side"]].astype(str),
        "barcode": dfm[MANIFEST_COLS["barcode"]].astype(str),
    })
    df_dess["case_id"] = df_dess["subject"] + "_" + df_dess["visit"] + "_" + df_dess["side"]
    df_dess["dess_path"] = df_dess["barcode"].apply(lambda b: str(DESS_DIR / f"{b}.nii.gz"))
    df_dess = df_dess[df_dess["dess_path"].apply(lambda p: Path(p).exists())]
    print("ca tim thay qua manifest:", len(df_dess))
else:
    # fallback: chi dung 176 ca iMorphics da co san _0000.nii.gz
    rows = []
    for split, sub in [("Tr", D012/"imagesTr"), ("Ts", D012/"imagesTs")]:
        for p in sub.glob("*_0000.nii.gz"):
            cid = p.name.replace("_0000.nii.gz","")
            subj, visit, side = cid.split("_")
            rows.append(dict(subject=subj, visit=visit, side=side, case_id=cid, dess_path=str(p)))
    df_dess = pd.DataFrame(rows)
    print("KHONG co MANIFEST_FILE -> fallback 176 ca iMorphics. Tim thay:", len(df_dess))
df_dess.head()

KHONG co MANIFEST_FILE -> fallback 176 ca iMorphics. Tim thay: 176


,subject,visit,side,case_id,dess_path
0,9567704,V00,L,9567704_V00_L,/content/drive/MyDrive/nnUNet_raw/Dataset012_i...
1,9331465,V00,R,9331465_V00_R,/content/drive/MyDrive/nnUNet_raw/Dataset012_i...
2,9352437,V00,L,9352437_V00_L,/content/drive/MyDrive/nnUNet_raw/Dataset012_i...
3,9607698,V00,R,9607698_V00_R,/content/drive/MyDrive/nnUNet_raw/Dataset012_i...
4,9587749,V00,L,9587749_V00_L,/content/drive/MyDrive/nnUNet_raw/Dataset012_i...


## 3b) [MỚI] Mở rộng cohort bằng OAIZIB-CM (tối đa +507 ca) qua `subInfo`

Không cần `MANIFEST_FILE` cho phần này: `subInfo_train.xlsx` (+ `subInfo_test.xlsx`) đã chứa
`SubjectID` thật khớp trực tiếp với `CMT-ID` (dùng trong tên file `oaizib_{CMT-ID}_0000.nii.gz`).

Nguồn công bố OAI-ZIB xác nhận toàn bộ 507 ca đều là **đầu gối phải, visit baseline** →
`case_id = {SubjectID}_V00_R` suy ra trực tiếp, không cần đoán.

**Lưu ý cho `S2`/`S3` sau này:** mask gốc trong `Dataset001_KneeOA/labelsTr` chỉ có **5 lớp**
(femur, femoral cart., tibia, med/lat tibial cart.) — **không có meniscus/patella**. Các ca này
sẽ không khớp tên với `Dataset020/labelsTr` hiện có (đặt tên theo `subject_visit_side` của
iMorphics) nên sẽ rơi vào `mask_status = "need_inference"` — `S2` sẽ chạy nnU-Net 8-lớp đầy đủ
cho case này. Nếu muốn giữ ground-truth thật (5 lớp) thay vì để AI dự đoán lại từ đầu, `S2` cần
sửa thêm bước gộp (GT thật cho 5 lớp + AI chỉ cho patella/meniscus) — chưa nằm trong notebook
này, làm ở bước sau.

In [9]:
def load_subinfo(path):
    df = pd.read_excel(path)
    df["CMT-ID"] = df["CMT-ID"].astype(str).str.zfill(3)  # "1" -> "001" (khop ten file oaizib_XXX)
    df["side"] = "R"          # toan bo OAIZIB-CM la dau goi phai (Ambellan et al. 2019)
    df["visit"] = "V00"       # toan bo la baseline
    df["subject"] = df["SubjectID"].astype(str)
    df["case_id"] = df["subject"] + "_" + df["visit"] + "_" + df["side"]
    return df

assert SUBINFO_TRAIN.exists(), (
    f"KHONG THAY {SUBINFO_TRAIN} — day la file bat buoc cho buoc 3b. "
    "Neu ban chua tai len Drive, sua duong dan SUBINFO_TRAIN o cell Config."
)
frames = [load_subinfo(SUBINFO_TRAIN)]
if SUBINFO_TEST is not None and Path(SUBINFO_TEST).exists():
    frames.append(load_subinfo(SUBINFO_TEST))
    print("da nap ca subInfo_test.xlsx")
else:
    print("CANH BAO: khong co subInfo_test.xlsx -> chi nap duoc phan train (~404 ca), "
          "thieu ~103 ca test cua OAIZIB-CM. Khong chan pipeline, chi la cohort nho hon du kien.")

df_subinfo = pd.concat(frames, ignore_index=True)
print("tong ca OAIZIB-CM co subject that (subInfo):", len(df_subinfo))

def find_oaizib_image(cmt_id):
    for split in ["imagesTr", "imagesTs"]:
        p = D001 / split / f"oaizib_{cmt_id}_0000.nii.gz"
        if p.exists():
            return p
    return None

assert D001.exists(), f"KHONG THAY {D001} — kiem tra lai da tai/giai nen OAIZIB-CM vao dung RAW chua."
df_subinfo["dess_path"] = df_subinfo["CMT-ID"].apply(find_oaizib_image)
n_missing = df_subinfo["dess_path"].isna().sum()
print(f"khong tim thay anh cho {n_missing}/{len(df_subinfo)} ca "
      f"-> kiem tra da tai Dataset001_KneeOA/imagesTr,imagesTs chua")
if n_missing:
    print("vi du CMT-ID thieu anh:", df_subinfo[df_subinfo['dess_path'].isna()]['CMT-ID'].head(5).tolist())
df_subinfo = df_subinfo.dropna(subset=["dess_path"]).copy()
df_subinfo["dess_path"] = df_subinfo["dess_path"].astype(str)

# Gop vao df_dess da co (176 iMorphics tu buoc 3) — bo trung case_id neu co (uu tien nguon co truoc)
before = len(df_dess)
df_dess = pd.concat(
    [df_dess, df_subinfo[["subject","visit","side","case_id","dess_path"]]],
    ignore_index=True
).drop_duplicates(subset="case_id", keep="first")
after = len(df_dess)
print(f"df_dess: {before} -> {after} ca (+{after - before} tu OAIZIB-CM, sau khi bo trung case_id)")
df_dess.tail()

da nap ca subInfo_test.xlsx
tong ca OAIZIB-CM co subject that (subInfo): 507
khong tim thay anh cho 0/507 ca -> kiem tra da tai Dataset001_KneeOA/imagesTr,imagesTs chua
df_dess: 176 -> 673 ca (+497 tu OAIZIB-CM, sau khi bo trung case_id)


,subject,visit,side,case_id,dess_path
678,9695135,V00,R,9695135_V00_R,/content/drive/MyDrive/nnUNet_raw/Dataset001_K...
679,9700450,V00,R,9700450_V00_R,/content/drive/MyDrive/nnUNet_raw/Dataset001_K...
680,9709257,V00,R,9709257_V00_R,/content/drive/MyDrive/nnUNet_raw/Dataset001_K...
681,9750090,V00,R,9750090_V00_R,/content/drive/MyDrive/nnUNet_raw/Dataset001_K...
682,9987407,V00,R,9987407_V00_R,/content/drive/MyDrive/nnUNet_raw/Dataset001_K...


## 4) Ghép cohort cuối: KL grade ⨝ ảnh DESS, đánh dấu ca đã có mask

In [10]:
assert len(df_kl) > 0, (
    "df_kl rong -> khong the ghep cohort. Kiem tra lai Cell 1 (parse_oai_code) va "
    "KL_ID_COL/KL_SIDE_COL/KL_GRADE_COL co dung ten cot khong."
)
if not {"case_id", "dess_path"}.issubset(df_dess.columns):
    raise RuntimeError(
        "df_dess dang rong hoac thieu cot \'case_id\'/\'dess_path\' -> khong the ghep. "
        "Nguyen nhan thuong gap: D012 (Dataset012_iMorphics) chua ton tai (xem lai RAW o "
        "cell Config phia tren), D001 (Dataset001_KneeOA) chua ton tai cho buoc 3b, hoac "
        "subInfo_train.xlsx thieu/sai duong dan."
    )

cohort = df_kl.merge(df_dess[["case_id", "dess_path"]], on="case_id", how="inner")
cohort["mask_status"] = cohort["case_id"].apply(lambda c: "ready" if c in set(done_case_ids) else "need_inference")

print("tong ca co CA KL grade LAN anh DESS:", len(cohort))
print(cohort["mask_status"].value_counts())
print(cohort["KL"].value_counts().sort_index())

out_csv = OUT_DIR / "cohort_manifest.csv"
cohort.to_csv(out_csv, index=False)
print("da luu:", out_csv)
cohort.head()

tong ca co CA KL grade LAN anh DESS: 559
mask_status
need_inference    492
ready              67
Name: count, dtype: int64
KL
0    103
1     60
2    137
3    183
4     76
Name: count, dtype: int64
da luu: /content/drive/MyDrive/knee_biomarkers/cohort_manifest.csv


,subject,side,visit,KL,case_id,dess_path,mask_status
0,9001104,R,V00,3,9001104_V00_R,/content/drive/MyDrive/nnUNet_raw/Dataset001_K...,need_inference
1,9002430,R,V00,2,9002430_V00_R,/content/drive/MyDrive/nnUNet_raw/Dataset001_K...,need_inference
2,9002817,R,V00,3,9002817_V00_R,/content/drive/MyDrive/nnUNet_raw/Dataset001_K...,need_inference
3,9003406,L,V00,2,9003406_V00_L,/content/drive/MyDrive/nnUNet_raw/Dataset012_i...,need_inference
4,9003430,R,V00,1,9003430_V00_R,/content/drive/MyDrive/nnUNet_raw/Dataset001_K...,need_inference


## Ghi chú
- File `cohort_manifest.csv` là input trực tiếp cho **S2** (chạy inference cho các case
  `mask_status == "need_inference"`, còn `"ready"` thì copy thẳng mask từ `Dataset020`).
- **[CẬP NHẬT] Cỡ mẫu kỳ vọng sau bản này:** tối đa 176 (iMorphics) + 507 (OAIZIB-CM, qua bước
  3b) = 683 ca trước khi lọc theo KL hợp lệ; số thực tế sau `inner join` với `KXR_SQ_BU00.txt`
  thường thấp hơn (không phải case nào cũng có KL hợp lệ, và một số subject có thể trùng ở cả
  2 nguồn — đã được `drop_duplicates` xử lý, ưu tiên giữ bản ghi iMorphics nếu trùng `case_id`).
- Nếu vẫn thấy `cohort` quá nhỏ so với 683: kiểm tra in ra `n_missing` ở bước 3b (thiếu ảnh
  OAIZIB-CM) và số case bị `df_kl` loại do không có KL hợp lệ ở `KXR_SQ_BU00.txt`.
- **Chưa dùng được** để mở rộng thêm ở bản này: pool `OAI_DESS` rộng hơn (barcode từ `label.csv`,
  ~4.600 subject khác) — vì (1) chưa tải ảnh full-res tương ứng, chỉ có bản `.npz` đã resize
  `(120,160,160)` KHÔNG có spacing/affine, không dùng được cho segmentation/biomarker theo đúng
  tỷ lệ thật; (2) `label.csv` không có cột `visit`. Khi giải quyết được 2 điểm này (tải thêm ảnh
  full-res + xác định visit qua file tracking gốc của OAI), quay lại điền `MANIFEST_FILE` ở bước 3
  để mở rộng cohort lần nữa.
- Mask của OAIZIB-CM (bước 3b) chỉ có 5 lớp (femur/tibia/2 sụn chày/sụn đùi) — xem ghi chú ở
  bước 3b về việc `S2` cần sửa thêm nếu muốn giữ ground-truth thật thay vì để AI dự đoán lại
  toàn bộ 8 lớp cho các ca này.
- Cỡ mẫu quyết định trực tiếp việc bạn có nên tiếp tục sang S2 (compute-heavy: infer hàng loạt)
  hay nên đầu tư thêm thời gian tìm ảnh/manifest trước.


## 5) [MỚI] Đánh dấu nguồn gốc nhãn (`source_dataset`) — phục vụ QC/sensitivity analysis ở S2-S5

Case trong `cohort` giờ đến từ đúng 2 nguồn (theo bước 3 + 3b ở trên):
- **iMorphics** (`case_id` khớp file trong `Dataset012_iMorphics/images{Tr,Ts}`) → nhãn 8-lớp
  **chuyên gia thật 100%** → `gt_real_imorphics`.
- **OAIZIB-CM** (`case_id` khớp `df_subinfo`, tức đã tìm được ảnh qua `subInfo_*.xlsx`) → nhãn
  1-5 (femur/tibia/2 sụn chày/sụn đùi) là **GT thật**, còn 6-8 (meniscus/patellar cart.) sẽ luôn
  là AI/pseudo (dùng đúng tên `partial_gt_oaizib_ai_meniscus_patella` — khớp tên S2 dùng ở bước
  1b/7, để 2 file luôn đồng bộ 1 tên gọi cho cùng 1 mức tin cậy, dù case đó `ready` hay
  `need_inference`).

Cột này **không đổi gì về pipeline mask/biomarker** — chỉ là bookkeeping để S3 gắn vào bảng
biomarker, và S4/S5 dùng làm sensitivity analysis (so kết quả trên tập tin cậy cao vs toàn bộ).

In [11]:
imorphics_ids = set()
for sub in ["imagesTr", "imagesTs"]:
    p = D012 / sub
    if p.exists():
        imorphics_ids |= {f.name.replace("_0000.nii.gz", "") for f in p.glob("*_0000.nii.gz")}

oaizib_ids = set(df_subinfo["case_id"]) if "df_subinfo" in dir() else set()

def _source_dataset(row):
    if row["case_id"] in imorphics_ids:
        return "gt_real_imorphics"
    if row["case_id"] in oaizib_ids:
        return "partial_gt_oaizib_ai_meniscus_patella"
    return "unknown_source"   # khong nen xay ra, nhung khong de crash pipeline neu co nguon khac sau nay

cohort["source_dataset"] = cohort.apply(_source_dataset, axis=1)
print(cohort["source_dataset"].value_counts())

if (cohort["source_dataset"] == "unknown_source").any():
    print("\nCANH BAO: co case khong khop imorphics_ids lan oaizib_ids -> kiem tra lai nguon du lieu:")
    print(cohort.loc[cohort["source_dataset"] == "unknown_source", "case_id"].head(10).tolist())

# ghi de lai cohort_manifest.csv voi cot moi (S2/S3/S4/S5 doc lai file nay)
cohort.to_csv(out_csv, index=False)
print("\nda cap nhat (them source_dataset):", out_csv)


source_dataset
partial_gt_oaizib_ai_meniscus_patella    471
gt_real_imorphics                         88
Name: count, dtype: int64

da cap nhat (them source_dataset): /content/drive/MyDrive/knee_biomarkers/cohort_manifest.csv
